In [1]:
import random
import pandas as pd
import numpy as np

# -----------------------------
# CONFIG
# -----------------------------
N_SAMPLES = 8000

food_base_time = {
    "rice_meal": 120,
    "biryani": 115,
    "fried_rice": 125,
    "dry_food_roti": 180,
    "bread_items": 170,
    "fried_snacks": 140,
    "samosa_kachori": 135,
    "curry_gravy": 100,
    "dal": 110,
    "paneer_dish": 95,
    "chicken_curry": 90,
    "fish_curry": 80,
    "noodles": 110,
    "pasta": 105,
    "dairy_sweets": 60,
    "milk_based_dessert": 55,
    "salad_cut_fruits": 70,
    "raw_vegetable_salad": 75,
    "bakery_cake": 85,
    "pizza": 95
}

packaging_penalty_range = {
    "sealed": (5, 10),
    "semi_covered": (15, 25),
    "open": (30, 45)
}

sun_exposure_temp_bonus = {
    "indoor_ac": (-4, -1),
    "indoor_no_ac": (0, 2),
    "outdoor_shaded": (2, 5),
    "outdoor_direct_sun": (5, 9)
}

data_rows = []

for _ in range(N_SAMPLES):

    food_type = random.choice(list(food_base_time.keys()))
    base_time = food_base_time[food_type]

    # environmental conditions
    temperature = round(random.uniform(24, 42), 1)
    humidity = round(random.uniform(35, 95), 1)

    time_since_cooked = random.randint(5, 150)

    packaging = random.choice(list(packaging_penalty_range.keys()))
    sun_exposure = random.choice(list(sun_exposure_temp_bonus.keys()))

    quantity_kg = round(random.uniform(3, 25), 1)

    # -----------------------------
    # EFFECTIVE TEMPERATURE
    # -----------------------------
    temp_bonus = random.uniform(*sun_exposure_temp_bonus[sun_exposure])
    effective_temp = temperature + temp_bonus

    # -----------------------------
    # DECAY COMPONENTS
    # -----------------------------
    temp_penalty = max(0, (effective_temp - 25)) * random.uniform(1.2, 2.2)
    humidity_penalty = max(0, (humidity - 50)) * random.uniform(0.3, 0.9)

    packaging_penalty = random.uniform(*packaging_penalty_range[packaging])

    # thermal mass effect (large quantity retains heat → more spoilage)
    quantity_penalty = quantity_kg * random.uniform(0.4, 0.9)

    # food risk adjustment
    high_risk_foods = ["fish_curry", "dairy_sweets", "milk_based_dessert"]
    if food_type in high_risk_foods:
        food_risk_penalty = random.uniform(8, 18)
    else:
        food_risk_penalty = random.uniform(0, 8)

    # -----------------------------
    # SAFE TIME CALCULATION
    # -----------------------------
    safe_remaining = (
        base_time
        - temp_penalty
        - humidity_penalty
        - packaging_penalty
        - quantity_penalty
        - food_risk_penalty
        - time_since_cooked
    )

    # add real-world noise
    noise = random.uniform(-8, 8)
    safe_remaining = safe_remaining + noise

    safe_remaining = max(0, round(safe_remaining, 1))

    data_rows.append([
        food_type,
        base_time,
        temperature,
        humidity,
        time_since_cooked,
        packaging,
        sun_exposure,
        quantity_kg,
        safe_remaining
    ])

columns = [
    "food_type",
    "base_safe_time",
    "temperature_c",
    "humidity_percent",
    "time_since_cooked_min",
    "packaging_type",
    "sun_exposure",
    "quantity_kg",
    "safe_minutes_remaining"
]

df = pd.DataFrame(data_rows, columns=columns)

df.to_csv("food_spoilage_dataset.csv", index=False)

print("✅ Dataset generated with shape:", df.shape)
print(df.head())

✅ Dataset generated with shape: (8000, 9)
          food_type  base_safe_time  temperature_c  humidity_percent  \
0  salad_cut_fruits              70           26.7              42.2   
1        fried_rice             125           40.7              77.2   
2      fried_snacks             140           30.5              48.6   
3       paneer_dish              95           41.1              41.1   
4           noodles             110           35.5              88.6   

   time_since_cooked_min packaging_type    sun_exposure  quantity_kg  \
0                     88         sealed  outdoor_shaded          4.6   
1                      5         sealed  outdoor_shaded         22.6   
2                     56         sealed  outdoor_shaded         20.0   
3                     89         sealed    indoor_no_ac         16.6   
4                    122   semi_covered       indoor_ac         24.1   

   safe_minutes_remaining  
0                     0.0  
1                    44.5  
2       

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("food_spoilage_dataset.csv")

In [4]:
df.head()

,food_type,base_safe_time,temperature_c,humidity_percent,time_since_cooked_min,packaging_type,sun_exposure,quantity_kg,safe_minutes_remaining
0,salad_cut_fruits,70,26.7,42.2,88,sealed,outdoor_shaded,4.6,0.0
1,fried_rice,125,40.7,77.2,5,sealed,outdoor_shaded,22.6,44.5
2,fried_snacks,140,30.5,48.6,56,sealed,outdoor_shaded,20.0,41.3
3,paneer_dish,95,41.1,41.1,89,sealed,indoor_no_ac,16.6,0.0
4,noodles,110,35.5,88.6,122,semi_covered,indoor_ac,24.1,0.0


In [9]:
df.drop(columns=["food_type","packaging_type","sun_exposure"]).corr()

,base_safe_time,temperature_c,humidity_percent,time_since_cooked_min,quantity_kg,safe_minutes_remaining
base_safe_time,1.000000,0.004698,-0.010459,0.021519,0.002819,0.501999
temperature_c,0.004698,1.000000,-0.006100,0.006897,0.000413,-0.107810
humidity_percent,-0.010459,-0.006100,1.000000,0.027292,0.016783,-0.123580
time_since_cooked_min,0.021519,0.006897,0.027292,1.000000,-0.012507,-0.476030
quantity_kg,0.002819,0.000413,0.016783,-0.012507,1.000000,-0.050090
safe_minutes_remaining,0.501999,-0.107810,-0.123580,-0.476030,-0.050090,1.000000


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb
import joblib


df = pd.read_csv("food_spoilage_dataset.csv")

print("Dataset shape:", df.shape)


cat_cols = ["food_type", "packaging_type", "sun_exposure"]

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le   # save for inference


TARGET = "safe_minutes_remaining"

X = df.drop(columns=[TARGET])
y = df[TARGET]


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nModel Performance")
print("MAE:", round(mae,2))
print("RMSE:", round(rmse,2))
print("R2 Score:", round(r2,3))

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop Features:")
print(importance)


joblib.dump(model, "spoilage_lightgbm.pkl")
joblib.dump(encoders, "label_encoders.pkl")


Dataset shape: (8000, 9)
Train size: (6400, 8)
Test size: (1600, 8)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001093 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 848
[LightGBM] [Info] Number of data points in the train set: 6400, number of used features: 8
[LightGBM] [Info] Start training from score 9.748078
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Lig

['label_encoders.pkl']

In [9]:
import joblib
import pandas as pd


model = joblib.load("spoilage_lightgbm.pkl")
encoders = joblib.load("label_encoders.pkl")


input_data = {
    "food_type": "dry_food_roti",
    "base_safe_time": 180,
    "temperature_c": 27,
    "humidity_percent": 45,
    "time_since_cooked_min": 10,
    "packaging_type": "sealed",
    "sun_exposure": "indoor_ac",
    "quantity_kg": 5

}


df = pd.DataFrame([input_data])

for col, le in encoders.items():
    df[col] = le.transform(df[col])


_pred_safe_minutes = model.predict(df)
pred_safe_minutes = _pred_safe_minutes[0]
print(pred_safe_minutes)
if pred_safe_minutes > 90:
    risk = "LOW"
elif pred_safe_minutes > 45:
    risk = "MEDIUM"
else:
    risk = "HIGH"

print("\n✅ Predicted Safe Time:", round(pred_safe_minutes,2), "minutes")
print("⚠️ Risk Level:", risk)

155.9291966066789

✅ Predicted Safe Time: 155.93 minutes
⚠️ Risk Level: LOW


In [3]:
import joblib
encoders = joblib.load("label_encoders.pkl")
encoders.items()

dict_items([('food_type', LabelEncoder()), ('packaging_type', LabelEncoder()), ('sun_exposure', LabelEncoder())])

df

In [11]:
import pandas as pd

df = pd.read_csv("food_spoilage_dataset.csv")

In [12]:
df["food_type"].unique()

<StringArray>
[   'salad_cut_fruits',          'fried_rice',        'fried_snacks',
         'paneer_dish',             'noodles',  'milk_based_dessert',
       'chicken_curry',      'samosa_kachori',         'bakery_cake',
        'dairy_sweets',       'dry_food_roti',               'pasta',
             'biryani', 'raw_vegetable_salad',         'curry_gravy',
           'rice_meal',               'pizza',          'fish_curry',
         'bread_items',                 'dal']
Length: 20, dtype: str

In [13]:
df.columns

Index(['food_type', 'base_safe_time', 'temperature_c', 'humidity_percent',
       'time_since_cooked_min', 'packaging_type', 'sun_exposure',
       'quantity_kg', 'safe_minutes_remaining'],
      dtype='str')